# Chapter 2 — Signals and Signal Space

Computer exercises from Section 2.10 of Lathi & Ding: basic signal generation and graphing, signal operations, periodic signals and power, signal correlation, and numerical Fourier-series coefficients.

```{admonition} Running these exercises
:class: tip
Every figure on this page is produced by the code directly above it, and the
code runs when the site is built. Use the **launch button** (the rocket icon) at the
top of the page to open this notebook in Google Colab or a live Binder session,
or hit **live code** to run and edit the cells right here in the browser.
```


In [ ]:
# Standard imports used by every exercise on this page.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad


## 2.10.1 Basic signals and signal graphing

Three building-block signals appear throughout the course, so we define them
once here and reuse them in every later exercise on this page:

| Function | Symbol | Definition |
|---|---|---|
| `ustep(t)` | $u(t)$ | $1$ for $t \ge 0$, else $0$ |
| `rect(t)` | $\mathrm{rect}(t)$ | $1$ for $|t| < 0.5$, else $0$ |
| `triangl(t)` | $\Delta(t)$ | $1 - |t|$ for $|t| < 1$, else $0$ |

Each one takes a NumPy array of times and returns an array of the same shape,
which is what lets us build more complicated signals by multiplying and
subtracting them.


In [ ]:
# Unit step u(t): 0 before t = 0, 1 from t = 0 onward.
# Comparing an array against 0 gives booleans; casting to float turns them
# into the 0.0/1.0 values we want to multiply other signals by.

def ustep(t):
    y = np.array(t>=0, dtype=float)
    return y

# Unit rectangle rect(t): 1 on |t| < 0.5, 0 outside.
# Built as the difference of two steps, one rising at -0.5 and one at +0.5.

def rect(t):
    y = np.array((np.sign(t+0.5)-np.sign(t-0.5)) > 0, dtype=float)
    return y

# Unit triangle in the textbook's notation: peaks at 1 when t = 0 and
# falls linearly to 0 at t = +/-1. The (t >= -1) & (t < 1) mask zeroes it
# outside that window.

def triangl(t):
    y = np.array((1 - np.abs(t)) * ((t >= -1) & (t < 1)), dtype=float)
    return y

### Graphing a signal

Any signal plot needs two arrays: the times $t$, and the signal value at each
of those times. Here we build

$$y(t) = e^{-t}\sin(8\pi t)\,u(t+1)$$

The $u(t+1)$ factor switches the signal on at $t = -1$, so everything to the
left of that is exactly zero. The result is Figure 1.


In [ ]:
# A signal plot is two arrays: the time axis, and the value at each time.
t = np.arange(-2, 3, 0.01)      # times from -2 to 3, sampled every 0.01 s

# Evaluate y(t) = exp(-t) sin(8*pi*t) u(t+1) at every one of those times.
# The ustep(t+1) factor forces y to be exactly 0 before t = -1.
y = np.exp(-t) * np.sin(8 * np.pi * t) * ustep(t+1)


fig1 = plt.figure()
plt.plot(t, y, linewidth=2, color="purple")

# Axis labels use mathtext (the r'$...$' strings) so they typeset properly.
plt.xlabel(r'$t$')
plt.ylabel(r'$y(t)$')
plt.title(r'$y(t) = e^{-t}\sin(8\pi t)u(t+1)$', fontsize=14)
plt.grid()
plt.show()

**Figure 1** — $y(t) = e^{-t}\sin(8\pi t)u(t+1)$. The step turns the
signal on at $t = -1$; the exponential envelope decays from there.


## 2.10.2 Signal operations

Time scaling, shifting, and inversion (Sec. 2.3) are all just changes to the
*argument* of the signal. Starting from

$$y(t) = e^{-|t|/4}\left[u(t) - u(t-4)\right]$$

we form

$$y_1(t) = y(2t) \qquad y_2(t) = y(t+2) \qquad y_3(t) = y(2-2t)$$

The key idea in the code below: we never rebuild the signal from scratch. We
build a *new time vector*, evaluate the same expression on it, and plot the
result against the original axis. $y_3$ combines all three operations at once.
All four appear in Figure 2.


In [ ]:
# Every transformation below works the same way: build a new time vector,
# evaluate the SAME expression on it, then plot against the original axis.
t = np.arange(-3, 5.00, 0.002)

# The original signal, y(t) = exp(-|t|/4) [u(t) - u(t-4)]:
# a decaying exponential gated to the window 0 <= t < 4.
y = np.exp(-np.abs(t)/4) * (ustep(t) - ustep(t-4))
fig, axs = plt.subplots(2, 2, figsize=(10, 8))

axs[0, 0].plot(t, y, linewidth=2, color='b')
axs[0, 0].set_xlabel(r'$t$')
axs[0, 0].set_ylabel(r'$y(t)$')
axs[0, 0].set_title('original signal y(t)')
axs[0, 0].grid()

linec = "purple"

# TIME SCALING: y_1(t) = y(2t). Feeding 2t into the signal makes it run twice
# as fast, so the pulse is squeezed to half its original width.
t1 = t * 2
y1 = np.exp(-np.abs(t1)/4) * (ustep(t1) - ustep(t1-4))
axs[0, 1].plot(t, y1, linewidth=2, color=linec)
axs[0, 1].set_xlabel(r'$t$')
axs[0, 1].set_ylabel(r'$y_1(t)$')
axs[0, 1].set_title('time scaling y(2t)')
axs[0, 1].grid()

# TIME SHIFTING: y_2(t) = y(t+2). Adding to the argument moves the signal
# EARLIER (to the left) by 2 seconds -- a common sign trap.
t2 = t + 2
y2 = np.exp(-np.abs(t2)/4) * (ustep(t2) - ustep(t2-4))
axs[1, 0].plot(t, y2, linewidth=2, color=linec)
axs[1, 0].set_xlabel(r'$t$')
axs[1, 0].set_ylabel(r'$y_2(t)$')
axs[1, 0].set_title('time shifting y(t+2)')
axs[1, 0].grid()

# ALL THREE AT ONCE: y_3(t) = y(2-2t). The negative coefficient on t
# reverses the signal, the factor 2 compresses it, and the constant shifts it.
t3 = 2 - t * 2
y3 = np.exp(-np.abs(t3)/4) * (ustep(t3) - ustep(t3-4))
axs[1, 1].plot(t, y3, linewidth=2, color=linec)
axs[1, 1].set_xlabel(r'$t$')
axs[1, 1].set_ylabel(r'$y_3(t)$')
axs[1, 1].set_title('y(2-2t)')
axs[1, 1].grid()

# Same axes on every panel, so the shapes can be compared directly.
for ax in axs.flat:
    ax.set_xlim([-3, 5])
    ax.set_ylim([-0.5, 1.5])

fig.tight_layout(pad=1.0)

plt.show()

**Figure 2** — the original signal and three transformations of it.
Compare each panel's time axis against the original to see what the operation
did: $y(2t)$ compresses, $y(t+2)$ shifts left, $y(2-2t)$ reverses *and*
compresses *and* shifts.


## 2.10.3 Periodic signals and signal power

To build a periodic signal, define one period and then repeat it. The code
below generates $2M$ periods of a signal with period $T = 6$, then measures
two things you should be able to predict from the plot:

- **average power** `y_power` — the mean of $y^2(t)$ over the whole record
- **energy in one period** `y_energyT` — the integral of $y^2(t)$ over $[0, T]$

Both are computed as Riemann sums: multiply by the sample spacing `Dt` to turn
a sum into an integral.


In [ ]:
# Build ONE period, then repeat it -- that is all a periodic signal is.
Dt = 0.002                  # sample spacing (seconds)
T = 6                       # period
M = 3                       # generate 2M periods total
t = np.arange(0, T, Dt)     # one period's worth of time samples

# The signal within a single period: a damped sinusoid, gated off after t = 4,
# so each period ends with 2 seconds of silence.
y = np.exp(-np.abs(t) / 2) * np.sin(2 * np.pi * t) * (ustep(t) - ustep(t - 4))

# Tile that period across the time axis, offsetting each copy by i*T.
time = np.array([])
y_periodic = np.array([])
for i in range(-M, M):
    time = np.concatenate((time, i * T + t))
    y_periodic = np.concatenate((y_periodic, y))

plt.figure()
fy = plt.plot(time, y_periodic)
plt.setp(fy, linewidth=2,color='purple')
plt.ylim(-1, 1)
plt.xlabel(r'$t$')
plt.grid()
plt.show()

# Average power = mean of y^2 over the record. Multiplying the sum by Dt
# turns it into an integral; dividing by the duration makes it an average.
y_power = np.sum(y_periodic ** 2) * Dt / (max(time) - min(time))
print("y_power = {:.8f}".format(y_power))

# Energy in a single period = integral of y^2 over [0, T].
# Consistency check: this should equal y_power * T.
y_energyT = np.sum(y ** 2) * Dt
print("y_energy_in_T = {:.8f}".format(y_energyT))

**Figure 3** — a periodic signal built by repeating one period six
times. The printed `y_power` and `y_energy_in_T` values come from the code
above, so they update if you change `T`, `M`, or the signal itself.


## 2.10.4 Signal correlation

The correlation coefficient measures how similar two signals are, independent
of their amplitude. For real signals,

$$c_n = \frac{1}{\sqrt{E_x E_{g_n}}}\int x(t)\,g_n(t)\,dt$$

Figure 4 shows $x(t)$ and five candidate signals $g_1(t) \dots g_5(t)$. Before
running the next cell, look at the plots and predict which will correlate most
strongly with $x(t)$, and which will come out near zero.


In [ ]:
# The reference signal x(t) and five candidates to compare against it.
Dt = 0.01                      # sample spacing
T = 6.0                        # record length
t = np.arange(-1, T+Dt, Dt)

# x(t): a rectangular pulse, on over 0 <= t < 5.
x = ustep(t) - ustep(t-5)
g1 = 0.5 * (ustep(t) - ustep(t-5))              # same shape, half amplitude
g2 = -(ustep(t) - ustep(t-5))                   # same shape, inverted
g3 = np.exp(-t/5) * (ustep(t) - ustep(t-5))     # slowly decaying
g4 = np.exp(-t) * (ustep(t) - ustep(t-5))       # rapidly decaying
g5 = np.sin(2 * np.pi * t) * (ustep(t) - ustep(t-5))  # 5 whole cycles

fig, axs = plt.subplots(2, 3)
fig.set_size_inches(14, 7)
fig.subplots_adjust(hspace=0.4, wspace=0.3)

axs[0, 0].plot(t, x, 'b', linewidth=2)
axs[0, 0].set_xlabel(r'$t$')
axs[0, 0].set_ylabel(r'${\bf x}({\it t })$')

axs[0, 1].plot(t, g1, 'b', linewidth=2)
axs[0, 1].set_xlabel(r'$t$')
axs[0, 1].set_ylabel(r'${\bf g_1}({\it t })$')

axs[0, 2].plot(t, g2, 'b', linewidth=2)
axs[0, 2].set_xlabel(r'$t$')
axs[0, 2].set_ylabel(r'${\bf g_2}({\it t })$')

axs[1, 0].plot(t, g3, linewidth=2,color='purple')
axs[1, 0].set_xlabel(r'$t$')
axs[1, 0].set_ylabel(r'${\bf g_3}({\it t })$')

axs[1, 1].plot(t, g4, linewidth=2,color='purple')
axs[1, 1].set_xlabel(r'$t$')
axs[1, 1].set_ylabel(r'${\bf g_4}({\it t })$')

axs[1, 2].plot(t, g5, linewidth=2,color='purple')
axs[1, 2].set_xlabel(r'$t$')
axs[1, 2].set_ylabel(r'${\bf g_5}({\it t })$')
# Identical axes everywhere so amplitudes can be compared by eye.
for ax in axs.flat:
    ax.set_xlim([-0.5, 6])
    ax.set_ylim([-1.2, 1.2])
    ax.grid()

plt.show()

**Figure 4** — the six signals of Example 2.6. $x(t)$ is the
reference; $g_1$ through $g_5$ are compared against it.


In [ ]:
# Energy of each signal: integral of |g(t)|^2, again as a Riemann sum.
E0 = np.sum(x * np.conj(x)) * Dt
E1 = np.sum(g1 * np.conj(g1)) * Dt
E2 = np.sum(g2 * np.conj(g2)) * Dt
E3 = np.sum(g3 * np.conj(g3)) * Dt
E4 = np.sum(g4 * np.conj(g4)) * Dt
E5 = np.sum(g5 * np.conj(g5)) * Dt

# Correlation coefficient: the inner product of the two signals, normalised
# by both energies. Dividing out the energies is what makes the answer
# independent of amplitude -- c is +1 for any positive multiple of x.
c0 = np.sum(x * np.conj(x)) * Dt/(np.sqrt(E0 * E0))
print("c0 = {:.4f}".format(c0))
c1 = np.sum(x * np.conj(g1)) * Dt/(np.sqrt(E0 * E1))
print("c1 = {:.4f}".format(c1))
c2 = np.sum(x * np.conj(g2)) * Dt/(np.sqrt(E0 * E2))
print("c2 = {:.4f}".format(c2))
c3 = np.sum(x * np.conj(g3)) * Dt/(np.sqrt(E0 * E3))
print("c3 = {:.4f}".format(c3))
c4 = np.sum(x * np.conj(g4)) * Dt/(np.sqrt(E0 * E4))
print("c4 = {:.4f}".format(c4))
c5 = np.sum(x * np.conj(g5)) * Dt/(np.sqrt(E0 * E5))
print('c5 = {:.4f}'.format(c5))

The results are worth reading carefully:

- $c_1 = 1$ — $g_1$ is $x$ at half the amplitude. Correlation ignores scale, so
  a *perfect* match is still $1$.
- $c_2 = -1$ — $g_2$ is $-x$: identical shape, opposite sign.
- $c_3 \approx 0.96$ — a slowly decaying exponential still looks a lot like a
  rectangular pulse.
- $c_4 \approx 0.63$ — decaying faster makes it less similar.
- $c_5 \approx 0$ — a sinusoid over a whole number of cycles is **orthogonal**
  to a constant. This is the property the entire Fourier series rests on.


## 2.10.5 Numerical computation of $D_n$

The exponential Fourier series coefficients

$$D_n = \frac{1}{T}\int_T g(t)\,e^{-jn\omega_0 t}\,dt$$

can be found two ways, and we do both so you can compare them.

**Method 1 — direct numerical integration.** Evaluate the integral above
literally, once per coefficient, using a quadrature routine. It is slow but it
follows the definition exactly. We use the triangle $\Delta(t)$ as the signal.


In [ ]:
# A scalar (not vectorised) triangle function, because the quadrature
# routine below feeds it one time value at a time.
# Peaks at 1 for t = 0 and falls to 0 at t = +/-1.
def funct_tri(t):
    y = int((t > -1) and (t < 1)) * (1 - np.absolute(t))
    return y

# scipy's quad only integrates real functions, but the Fourier integrand
# exp(-j n w0 t) g(t) is complex. Integrate the real and imaginary parts
# separately and recombine them.
def complex_quadrature(func, a, b, **kwargs):
    def real_func(x):
        return np.real(func(x))
    def imag_func(x):
        return np.imag(func(x))
    real_integral = quad(real_func, a, b, **kwargs)
    imag_integral = quad(imag_func, a, b, **kwargs)
    return real_integral[0] + 1j*imag_integral[0]

With $g(t)$ defined, we integrate for each $n$ from $-N$ to $N$.
Here the signal is $\Delta(t/2)$ on the period $[a, b] = [-2, 2]$, and
$N = 10$. Because $\Delta(t/2)$ is real and even, every $D_n$ should come out
real — the imaginary parts in Figure 5 are numerical noise at the $10^{-17}$
level, not signal.


In [ ]:
# METHOD 1: evaluate D_n = (1/T) * integral over one period of
# g(t) exp(-j n w0 t) dt, literally, one coefficient at a time.
a, b = -2, 2          # the period runs from a to b
tol = 1e-6            # absolute error tolerance for the quadrature
T = b - a             # period length
N = 10                # compute D_n for n = -N ... +N

# D is indexed so that D[N] holds D_0, with negative n below and positive above.
# Start with the n = 0 (DC) term, whose integrand has no exponential factor.
Func = lambda t: funct_tri(t/2)
D = np.zeros(2 * N + 1, dtype=complex)
D[N] = 1/T * complex_quadrature(Func, a, b, epsabs=tol)

for i in range(N):
    # Positive frequencies n = 1 ... N land in D[N+1] ... D[2N].
    Func = lambda t: np.exp(-1j*2*np.pi*t*(i+1)/T) * funct_tri(t/2)
    D[N+i+1] = 1/T * complex_quadrature(Func, a, b, epsabs=tol)

    # Negative frequencies n = -N ... -1 land in D[0] ... D[N-1].
    Func = lambda t: np.exp(1j*2*np.pi*t*(N-i)/T) * funct_tri(t/2)
    D[i] = 1/T * complex_quadrature(Func, a, b, epsabs=tol)
fig = plt.figure()
plt.subplot(211)
s1 = plt.stem(np.arange(-N, N+1), D.real)
plt.setp(s1, linewidth=2,color='purple')
plt.ylabel(r'Real($D_n$)')
plt.title(r'(a) Real part of $D_n$')
plt.grid()

plt.subplot(212)
s2 = plt.stem(np.arange(-N, N+1), np.imag(D))
plt.setp(s2, linewidth=2,color='blue')
plt.ylabel(r'Imag($D_n$)')
plt.title('(b) Imaginary part of $D_n$')
plt.grid();plt.ylim(-0.5, 0.5)

fig.tight_layout(pad=1.0)
plt.show()

**Figure 5** — Fourier coefficients $D_n$ of the repeated
triangle $\Delta(t/2)$, $T = 4$, by direct integration: (a) real part,
(b) imaginary part. The imaginary part is zero to within rounding error, as it
must be for a real even signal.


**Method 2 — via the FFT.** Sampling $g(t)$ and taking an FFT gets
all the coefficients at once, far faster than integrating one at a time.

The sampling details matter. Samples start at $t = 0$ and the last one is at
$t = T_0 - T_s$, not $T_0$ — the sample at $T_0$ would be a duplicate of the
one at $t = 0$. At a jump discontinuity the sample is the *average* of the two
sides, which is why the first sample below is $(e^{-\pi/2}+1)/2 = 0.604$ and
not $1$.

Choosing $N_0$: $D_n$ for a signal with a jump decays only as $1/n$, so we need
enough samples that the coefficients are small by $n = N_0/2$. With
$N_0 = 64$, $|D_{32}|$ is about $1\%$ of $|D_1|$, which is good enough. The
signal is the one from Lathi & Ding Fig. 2.19b (Example 2.7).


In [ ]:
# METHOD 2: sample the signal and let one FFT produce every coefficient.
T0 = np.pi            # period of the signal
N0 = 64               # samples per period (must resolve D_n out to n = N0/2)
Ts = T0/N0            # sampling interval
M = 12                # how many coefficients each side of DC to display

# Samples run from t = 0 to T0 - Ts. The sample at t = T0 is deliberately
# omitted: it would duplicate the one at t = 0 and double-count the period.
t = np.arange(0, Ts*N0, Ts)
g = np.exp(-t/2)

# At the jump discontinuity the correct sample value is the average of the two
# sides: (exp(-pi/2) + 1)/2 = 0.604, not 1.
g[0] = 0.604

# Dividing by N0 converts the FFT sum into the (1/T) integral of the
# Fourier series definition.
Gn = np.fft.fft(g)/N0
Gangle, Gmag = np.angle(Gn), np.absolute(Gn)

# The FFT returns n = 0, 1, ... N0-1. Negative frequencies live in the
# upper half of that array, so splice the top M-1 bins in front of the
# bottom M to get D_n running from -(M-1) to +(M-1) in natural order.
Dn=np.append(Gn[N0-M+1:N0],Gn[0:M])
Dnangle, Dnmag = np.angle(Dn), np.absolute(Dn)

n = np.arange(0, len(Dn))-(M-1)
fig = plt.figure()
plt.subplot(211)
s1 = plt.stem(n, Dnmag)
plt.setp(s1, linewidth=0.5,color='purple')
plt.ylabel(r'$|D_n|$')
plt.title(r'(a) Magnitude of $D_n$')
plt.grid()

plt.subplot(212)
s2 = plt.stem(n, Dnangle)
plt.setp(s2, linewidth=0.5,color='blue')
plt.ylabel(r'$<D_n$'); plt.xlabel(r'$n$')
plt.title('(b) Phase of $D_n$')
plt.grid();plt.ylim(-1.6, 1.6)
fig.tight_layout(pad=.3)
plt.show()

**Figure 6** — magnitude and phase of $D_n$ from the FFT method.

Converting the exponential series to the compact trigonometric form uses
$C_0 = |D_0|$ and $C_n = 2|D_n|$ for $n \ge 1$, with $\theta_n = \angle D_n$.


In [ ]:
# Convert the exponential series D_n into the compact trigonometric series.
# D_n and D_-n are a conjugate pair, so combining them doubles the amplitude:
# C_0 = |D_0| but C_n = 2|D_n| for n >= 1.
C0 = Gmag[0]
Cn = 2*Gmag[1:M]
Amplitudes=[C0]+list(Cn)
Angles=Gangle[0:M]*(180/np.pi)
print('Amplitudes |   Angles')
for (amplitude, angle) in zip(Amplitudes, Angles):
    print(" {0: <1.4f}    |  {1: < 3.4f}".format(amplitude, angle))

# C_n is the amplitude of each harmonic; theta_n is its phase in degrees.
xaxis=np.arange(len(Amplitudes))
plt.subplot(211); plt.ylabel(r'$C_n$')
plt.stem(xaxis,Amplitudes,'m'); plt.grid()
plt.subplot(212); plt.ylabel(r'$\theta_n$')
plt.stem(xaxis,Angles,'m');plt.grid()
plt.show()

**Figure 7** — the same signal as Figure 6, expressed as a compact
trigonometric series: amplitude $C_n$ (top) and phase $\theta_n$ in degrees
(bottom). Only non-negative $n$ appears here — the negative-frequency half of
the exponential series has been folded into it, which is why $C_n = 2|D_n|$.
